In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.patches import Patch

# ─── CONFIGURACIÓN DE RUTAS Y ESTILO ─────────────────────────────────────────
ROOT_DIR = Path("../../").resolve()
OUTPUT_DIR = ROOT_DIR / "reports" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# ─── 1. GRÁFICO DE CAJAS: PM2.5 ──────────────────────────────────────────────
def plot_pm25_boxplot(parquet_path):
    if not Path(parquet_path).exists():
        print(f"Archivo no encontrado: {parquet_path}")
        return

    df = pd.read_parquet(parquet_path)
    df['date'] = pd.to_datetime(df['date'])
    
    df['Periodo'] = 'Pre-ZBE'
    df.loc[df['date'] >= '2025-09-01', 'Periodo'] = 'Post-ZBE'

    data = df[['date', 'Periodo', 'PM2.5_zbe', 'PM2.5_out']].dropna()
    data_melted = data.melt(
        id_vars=['date', 'Periodo'], 
        value_vars=['PM2.5_zbe', 'PM2.5_out'], 
        var_name='Zona', 
        value_name='Concentración (µg/m³)'
    )
    data_melted['Zona'] = data_melted['Zona'].map({'PM2.5_zbe': 'Interior ZBE', 'PM2.5_out': 'Exterior ZBE'})

    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=data_melted, 
        x='Zona', 
        y='Concentración (µg/m³)', 
        hue='Periodo', 
        palette=['#bdc3c7', '#2ecc71'], 
        showfliers=False
    )
    plt.title('Distribución de PM2.5: Pre vs Post ZBE', fontsize=14, pad=15, fontweight='bold')
    plt.ylabel('Concentración (µg/m³)')
    plt.xlabel('')
    plt.legend(title='')
    plt.tight_layout()
    
    out_file = OUTPUT_DIR / '01_pm25_boxplot.png'
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Guardado: {out_file}")

# ─── 2. GRÁFICO DE BARRAS: PRUEBA DEL DOMINGO ────────────────────────────────
def plot_sunday_test():
    # Datos extraídos del log de la prueba del domingo
    categorias = ['Pre-ZBE', 'Post-ZBE\n(Bruto)', 'Post-ZBE\n(Ajustado clima)']
    valores = [13.60, 11.95, 11.18] # 13.60 - 2.42 (ajuste HDD) = 11.18
    colores = ['#95a5a6', '#e74c3c', '#27ae60']

    plt.figure(figsize=(8, 6))
    bars = plt.bar(categorias, valores, color=colores, width=0.5)
    
    plt.title('Impacto ZBE: NO$_2$ en Domingos de Invierno\n(Aislamiento de factor tráfico y clima)', fontsize=14, pad=15, fontweight='bold')
    plt.ylabel('Concentración NO$_2$ (µg/m³)')
    plt.ylim(0, 16)

    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.2, f'{yval:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Línea de tendencia visual del ajuste real
    plt.plot([0, 2], [valores[0], valores[2]], color='black', linestyle='--', alpha=0.4, linewidth=2)
    
    plt.tight_layout()
    out_file = OUTPUT_DIR / '02_prueba_domingo_no2.png'
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Guardado: {out_file}")

# ─── 3. GRÁFICO FEATURE IMPORTANCE ───────────────────────────────────────────
def plot_feature_importance():
    features = [
        'NO2_zbe', 'fc_temperature_2m_d1', 'exp_traffic_volume_d1', 
        'fc_wind_direction_10m_d1', 'PM10_zbe_roll_mean_14d', 
        'NO2_zbe_roll_mean_7d', 'exp_traffic_occupancy_d1', 
        'fc_wind_v_d1', 'NO2_zbe_roll_mean_3d', 'exp_traffic_volume_d3'
    ]
    importancia = [83.0, 56.0, 38.0, 31.0, 28.0, 28.0, 27.0, 23.0, 21.0, 20.0]

    # Nombres más cortos y directos
    feature_names_clean = {
        'NO2_zbe': 'NO$_2$ (Dato de hoy)',
        'fc_temperature_2m_d1': 'Temperatura (Mañana)',
        'exp_traffic_volume_d1': 'Volumen Tráfico (Mañana)',
        'fc_wind_direction_10m_d1': 'Dirección Viento (Mañana)',
        'PM10_zbe_roll_mean_14d': 'PM10 acumulado (14 días)',
        'NO2_zbe_roll_mean_7d': 'NO$_2$ acumulado (7 días)',
        'exp_traffic_occupancy_d1': 'Densidad Tráfico (Mañana)',
        'fc_wind_v_d1': 'Viento Norte-Sur (Mañana)',
        'NO2_zbe_roll_mean_3d': 'NO$_2$ acumulado (3 días)',
        'exp_traffic_volume_d3': 'Volumen Tráfico (A 3 días)'
    }

    # Agrupación por categorías para la leyenda
    categories = {
        'NO2_zbe': 'Contaminación Histórica',
        'fc_temperature_2m_d1': 'Meteorología',
        'exp_traffic_volume_d1': 'Tráfico',
        'fc_wind_direction_10m_d1': 'Meteorología',
        'PM10_zbe_roll_mean_14d': 'Contaminación Histórica',
        'NO2_zbe_roll_mean_7d': 'Contaminación Histórica',
        'exp_traffic_occupancy_d1': 'Tráfico',
        'fc_wind_v_d1': 'Meteorología',
        'NO2_zbe_roll_mean_3d': 'Contaminación Histórica',
        'exp_traffic_volume_d3': 'Tráfico'
    }

    # Colores por categoría
    color_map = {
        'Contaminación Histórica': '#e74c3c', # Rojo
        'Meteorología': '#3498db',            # Azul
        'Tráfico': '#f39c12'                  # Naranja
    }

    features_clean = [feature_names_clean.get(f, f) for f in features]
    colors = [color_map[categories[f]] for f in features]

    df_imp = pd.DataFrame({
        'Feature': features_clean, 
        'Importancia': importancia,
        'Color': colors
    })
    
    # Ordenar de menor a mayor para que el más importante quede arriba en el barh
    df_imp = df_imp.sort_values('Importancia', ascending=True)

    plt.figure(figsize=(10, 7))
    bars = plt.barh(df_imp['Feature'], df_imp['Importancia'], color=df_imp['Color'])
    
    plt.title('Top 10 Variables más influyentes (Modelo NO$_2$ intraZBE)', fontsize=14, pad=15, fontweight='bold')
    plt.xlabel('Importancia (Split/Gain)')
    
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 1, bar.get_y() + bar.get_height()/2, f'{width:.0f}', ha='left', va='center', fontsize=11)

    # Creación de la leyenda personalizada
    legend_elements = [Patch(facecolor=color_map[cat], label=cat) for cat in color_map]
    plt.legend(handles=legend_elements, loc='lower right', title="Tipo de Variable", frameon=True)

    plt.tight_layout()
    out_file = OUTPUT_DIR / '03_feature_importance_top10.png'
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Guardado: {out_file}")

# ─── EJECUCIÓN ───────────────────────────────────────────────────────────────
if __name__ == '__main__':
    parquet_path = ROOT_DIR / "data" / "processed" / "features_daily.parquet"
    
    print("Generando gráficos para el informe...")
    plot_pm25_boxplot(parquet_path)
    plot_sunday_test()
    plot_feature_importance()
    print("¡Proceso completado!")

Generando gráficos para el informe...
✅ Guardado: C:\Users\ortas\OneDrive\Documentos\Vitoria\vitoria-air-quality\reports\figures\01_pm25_boxplot.png
✅ Guardado: C:\Users\ortas\OneDrive\Documentos\Vitoria\vitoria-air-quality\reports\figures\02_prueba_domingo_no2.png
✅ Guardado: C:\Users\ortas\OneDrive\Documentos\Vitoria\vitoria-air-quality\reports\figures\03_feature_importance_top10.png
¡Proceso completado!
